# RAGAS Evaluation: context_precision + context_recall

Loads cached eval_data (from `query_rag_system.ipynb`) and runs context retrieval quality metrics.

**Metrics:**
- `context_precision` — Are the retrieved contexts relevant to the question?
- `context_recall` — Do the contexts cover the ground truth answer?

**Prerequisites:**
- `eval_data_health_wallet.json` generated by `query_rag_system.ipynb`

**See also:** `ragas_eval_answer_metrics.ipynb` for answer_similarity + answer_correctness.

## Setup

In [ ]:
import json
import time
from datetime import datetime

import pandas as pd
from llama_stack_client import LlamaStackClient
from rich.pretty import pprint

RAGAS_URL = "http://llama-stack-ragas-inline-service.ragas-eval.svc.cluster.local:8321"
PROVIDER_ID_INLINE = "trustyai_ragas_inline"

ragas_client = LlamaStackClient(base_url=RAGAS_URL)


def compute_aggregated(score_result):
    """Compute mean from per-question scores, skipping None/NaN entries.
    Falls back to RAGAS aggregated_results if available."""
    agg = score_result.aggregated_results
    if agg is not None and not (isinstance(agg, dict) and None in agg.values()):
        if isinstance(agg, dict):
            vals = [v for v in agg.values() if v is not None]
            return vals[0] if len(vals) == 1 else agg
        return agg
    scores = []
    for row in score_result.score_rows:
        s = row.get("score")
        if s is not None and str(s) != "nan":
            scores.append(float(s))
    return round(sum(scores) / len(scores), 6) if scores else None

In [ ]:
# Load eval_data from the main notebook's checkpoint
with open("eval_data_health_wallet.json", "r", encoding="utf-8") as f:
    eval_data = json.load(f)
print(f"Loaded {len(eval_data)} evaluation entries")

In [ ]:
# Find the LLM model ID in RAGAS system
ragas_models = ragas_client.models.list()
ragas_llm_model = None
for m in ragas_models:
    mid = getattr(m, 'identifier', None) or getattr(m, 'id', None)
    mtype = getattr(m, 'model_type', '')
    if hasattr(m, 'custom_metadata') and m.custom_metadata:
        mtype = m.custom_metadata.get('model_type', mtype)
    if mtype == 'llm':
        ragas_llm_model = mid
        break

print(f"LLM model: {ragas_llm_model}")

# Check eval providers
providers = ragas_client.providers.list()
eval_providers = [p for p in providers if p.api == 'eval']
pprint(eval_providers)

## Helper: Run a RAGAS Metric

In [ ]:
def run_ragas_metric(metric_names, eval_data, label=None):
    """Register dataset + benchmark, run eval, return results or None on failure."""
    label = label or "_".join(metric_names)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    dataset_id = f"hw_{label}_{ts}"
    benchmark_id = f"hw_bench_{label}_{ts}"

    # Register
    ragas_client.beta.datasets.register(
        dataset_id=dataset_id,
        purpose="eval/question-answer",
        source={"type": "rows", "rows": eval_data},
        metadata={"provider_id": "localfs"},
    )
    ragas_client.alpha.benchmarks.register(
        benchmark_id=benchmark_id,
        dataset_id=dataset_id,
        scoring_functions=metric_names,
        provider_id=PROVIDER_ID_INLINE,
    )

    # Run
    job = ragas_client.alpha.eval.run_eval(
        benchmark_id=benchmark_id,
        benchmark_config={
            "eval_candidate": {
                "type": "model",
                "model": ragas_llm_model,
                "sampling_params": {"temperature": 0.1, "max_tokens": 500},
            },
            "scoring_params": {},
        },
    )
    print(f"[{label}] Job {job.job_id} submitted. Metrics: {metric_names}")

    # Poll
    start = time.time()
    while True:
        st = ragas_client.alpha.eval.jobs.status(
            benchmark_id=benchmark_id, job_id=job.job_id
        )
        elapsed = time.time() - start
        print(f"  [{elapsed:.0f}s] {st.status}")
        if st.status in ("completed", "failed"):
            break
        time.sleep(15)

    if st.status == "failed":
        print(f"  FAILED after {elapsed:.0f}s")
        return None

    results = ragas_client.alpha.eval.jobs.retrieve(
        benchmark_id=benchmark_id, job_id=job.job_id
    )
    print(f"  Completed in {elapsed:.0f}s")
    for mn in metric_names:
        if mn in results.scores:
            print(f"  {mn}: {results.scores[mn].aggregated_results}")
    return results

## Run: context_precision + context_recall

In [ ]:
results_ctx = run_ragas_metric(
    ["context_precision", "context_recall"],
    eval_data,
    label="context",
)

## Combined Summary (all metrics)

In [ ]:
print("=" * 60)
print("ALL METRICS SUMMARY")
print("=" * 60)

print(f"  {'answer_similarity':25s}: (from main notebook)")
print(f"  {'answer_correctness':25s}: (from main notebook)")

if results_ctx:
    for mn in ["context_precision", "context_recall"]:
        if mn in results_ctx.scores:
            sr = results_ctx.scores[mn]
            agg = compute_aggregated(sr)
            scored = sum(1 for r in sr.score_rows if r.get("score") is not None and str(r.get("score")) != "nan")
            skipped = len(sr.score_rows) - scored
            suffix = f" ({skipped} skipped)" if skipped > 0 else ""
            print(f"  {mn:25s}: {agg}{suffix}")
else:
    print(f"  {'context_precision':25s}: FAILED")
    print(f"  {'context_recall':25s}: FAILED")

In [ ]:
# Per-question detail for successful metrics
all_results = {}
if results_ctx:
    for mn in ["context_precision", "context_recall"]:
        if mn in results_ctx.scores:
            all_results[mn] = results_ctx.scores[mn]

if all_results:
    rows = []
    ref_results = results_ctx
    for i, gen in enumerate(ref_results.generations):
        row = {"question": gen["user_input"][:70]}
        for mn, sr in all_results.items():
            if i < len(sr.score_rows):
                score = sr.score_rows[i].get("score", None)
                if score is not None and str(score) != 'nan':
                    row[mn] = round(score, 3)
                else:
                    row[mn] = "SKIP"
        rows.append(row)

    df = pd.DataFrame(rows)
    pd.set_option('display.max_colwidth', 70)
    pd.set_option('display.width', 200)
    print("\nPER-QUESTION SCORES:")
    print(df.to_string(index=False))

    # Count skipped entries
    for mn in all_results:
        if mn in df.columns:
            skips = (df[mn] == "SKIP").sum()
            if skips > 0:
                print(f"\n{mn}: {skips}/{len(df)} entries skipped (parsing failure)")

## Save Results

In [ ]:
import os

os.makedirs("results", exist_ok=True)

ctx_result = {}
if results_ctx:
    for mn in ["context_precision", "context_recall"]:
        m_data = {"metric": mn, "aggregated": None, "num_scored": 0, "num_skipped": 0, "per_question": []}
        if mn in results_ctx.scores:
            sr = results_ctx.scores[mn]
            m_data["aggregated"] = compute_aggregated(sr)
            for i, gen in enumerate(results_ctx.generations):
                score = sr.score_rows[i].get("score", None) if i < len(sr.score_rows) else None
                is_valid = score is not None and str(score) != "nan"
                if is_valid:
                    m_data["num_scored"] += 1
                else:
                    m_data["num_skipped"] += 1
                m_data["per_question"].append({
                    "question": gen["user_input"],
                    "score": round(float(score), 4) if is_valid else None,
                })
        ctx_result[mn] = m_data

    with open("results/context_metrics.json", "w", encoding="utf-8") as f:
        json.dump(ctx_result, f, ensure_ascii=False, indent=2)
    print("Saved results/context_metrics.json")
    for mn, md in ctx_result.items():
        print(f"  {mn}: mean={md['aggregated']}, scored={md['num_scored']}, skipped={md['num_skipped']}")
else:
    print("Context metrics not available (job failed) — nothing saved")